In [20]:
import pandas as pd
import numpy as np

# Load MIT-BIH Arrhythmia Dataset (Training and Test sets)
mitbih_train = pd.read_csv('mitbih_train.csv', header=None)
mitbih_test = pd.read_csv('mitbih_test.csv', header=None)

# Load PTB Diagnostic ECG Database (Normal and Abnormal samples)
ptbdb_normal = pd.read_csv('ptbdb_normal.csv', header=None)
ptbdb_abnormal = pd.read_csv('ptbdb_abnormal.csv', header=None)

# Separate features and labels for MIT-BIH Dataset
X_mitbih_train = mitbih_train.iloc[:, :-1].values  
y_mitbih_train = mitbih_train.iloc[:, -1].values   
X_mitbih_test = mitbih_test.iloc[:, :-1].values    
y_mitbih_test = mitbih_test.iloc[:, -1].values     




In [34]:
import tensorflow as tf
from tensorflow.keras.layers import Conv1D, MaxPooling1D, LSTM, Dense, Dropout, Input, Reshape, BatchNormalization
from tensorflow.keras.models import Model
from tensorflow.keras.regularizers import l2

# Define the CNN-LSTM 1D Model
def create_cnn_lstm_1d_model():
    inputs = Input(shape=(187, 1))  # Input shape is adjusted for 1D signals

    # Convolutional layers
    x = Conv1D(16, kernel_size=3, activation='relu')(inputs)
    x = MaxPooling1D(pool_size=2)(x)
    x = Dropout(0.25)(x)

    x = Conv1D(32, kernel_size=2, activation='relu')(x)
    x = MaxPooling1D(pool_size=2)(x)
    x = Dropout(0.25)(x)

    # LSTM layers
    x = LSTM(32, return_sequences=True)(x)
    x = LSTM(16)(x)

    # Dense layers
    x = Dense(64, activation='relu')(x)
    x = Dropout(0.5)(x)

    # Output layer
    outputs = Dense(5, activation='softmax')(x)  # 5 classes for MIT-BIH

    # Build the model
    model = Model(inputs=inputs, outputs=outputs)
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model


In [35]:
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE
from sklearn.metrics import classification_report
import numpy as np

# Initialize Stratified K-Fold
skf = StratifiedKFold(n_splits=3, random_state=42, shuffle=True)

fold_no = 1
batch_size = 512

final_predictions = []
final_true_labels = []

# Iterate through each fold
for train_index, val_index in skf.split(X_mitbih_train, y_mitbih_train):
    print(f"Training fold {fold_no}...")

    # Split the data into training and validation sets for this fold
    X_train_fold, X_val_fold = X_mitbih_train[train_index], X_mitbih_train[val_index]
    y_train_fold, y_val_fold = y_mitbih_train[train_index], y_mitbih_train[val_index]

    # Apply SMOTE to the training fold only
    smote = SMOTE(random_state=42)
    X_train_fold_smote, y_train_fold_smote = smote.fit_resample(X_train_fold.reshape(X_train_fold.shape[0], -1), y_train_fold)

    # Standardize data
    scaler = StandardScaler()
    X_train_fold_smote = scaler.fit_transform(X_train_fold_smote)
    X_val_fold = scaler.transform(X_val_fold.reshape(X_val_fold.shape[0], -1))

    # Reshape the data for 1D CNN
    X_train_fold_reshaped = X_train_fold_smote.reshape(-1, 187, 1)
    X_val_fold_reshaped = X_val_fold.reshape(-1, 187, 1)

    # Create the CNN-LSTM model
    model = create_cnn_lstm_1d_model()

    # Add early stopping
    early_stopping = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

    # Train the model
    history = model.fit(X_train_fold_reshaped, y_train_fold_smote,
                        epochs=10,
                        batch_size=batch_size,
                        validation_data=(X_val_fold_reshaped, y_val_fold),
                        callbacks=[early_stopping],
                        verbose=1)

    # Predict on the validation set for the current fold
    y_val_pred = model.predict(X_val_fold_reshaped)
    y_val_pred_classes = np.argmax(y_val_pred, axis=1)
    final_predictions.extend(y_val_pred_classes)
    final_true_labels.extend(y_val_fold)

    # Print classification report for the current fold
    print(f"Classification Report for Fold {fold_no}:\n", classification_report(y_val_fold, y_val_pred_classes, target_names=['N', 'S', 'V', 'F', 'Q']))

    # Clear the session to free up memory
    tf.keras.backend.clear_session()

    fold_no += 1

# Evaluate the final model on the MIT-BIH Test Set
X_mitbih_test = scaler.transform(X_mitbih_test.reshape(X_mitbih_test.shape[0], -1)).reshape(X_mitbih_test.shape[0], 187, 1)
y_test_pred = model.predict(X_mitbih_test)
y_test_pred_classes = np.argmax(y_test_pred, axis=1)

# Print classification report for the test set
print("Final Classification Report on MIT-BIH Test Set:\n", classification_report(y_mitbih_test, y_test_pred_classes, target_names=['N', 'S', 'V', 'F', 'Q']))


Training fold 1...
Epoch 1/10
472/472 ━━━━━━━━━━━━━━━━━━━━ 49s 93ms/step - accuracy: 0.5000 - loss: 1.2222 - val_accuracy: 0.6456 - val_loss: 0.9361
Epoch 2/10
472/472 ━━━━━━━━━━━━━━━━━━━━ 46s 97ms/step - accuracy: 0.7437 - loss: 0.7283 - val_accuracy: 0.6882 - val_loss: 0.8834
Epoch 3/10
472/472 ━━━━━━━━━━━━━━━━━━━━ 47s 99ms/step - accuracy: 0.8266 - loss: 0.5216 - val_accuracy: 0.7943 - val_loss: 0.6318
Epoch 4/10
472/472 ━━━━━━━━━━━━━━━━━━━━ 50s 105ms/step - accuracy: 0.8561 - loss: 0.4297 - val_accuracy: 0.7819 - val_loss: 0.6302
Epoch 5/10
472/472 ━━━━━━━━━━━━━━━━━━━━ 51s 109ms/step - accuracy: 0.8730 - loss: 0.3763 - val_accuracy: 0.8315 - val_loss: 0.4871
Epoch 6/10
472/472 ━━━━━━━━━━━━━━━━━━━━ 56s 120ms/step - accuracy: 0.8864 - loss: 0.3392 - val_accuracy: 0.8311 - val_loss: 0.5019
Epoch 7/10
472/472 ━━━━━━━━━━━━━━━━━━━━ 53s 113ms/step - accuracy: 0.8952 - loss: 0.3095 - val_accuracy: 0.7286 - val_loss: 0.7077
Epoch 8/10
472/472 ━━━━━━━━━━━━━━━━━━━━ 68s 145ms/step - accuracy: 

KeyboardInterrupt: 